<a href="https://colab.research.google.com/github/manishbgp007/Applied-AI-ML-Capstone-Project-Part-3/blob/main/Applied_AI_%26_ML_Capstone_Project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Upload the CSV file
from google.colab import files
import pandas as pd

# Load into DataFrame
df = pd.read_csv("/content/pd.read.csv")

# Load into DataFramecleaned_data
df = pd.read_csv("/content/cleaned_data.csv")

In [ ]:
# Classification Model (Logistic Regression)
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Target column
target_col = "median_house_value"

# Binary target bana lo, agar pehle se binary nahi hai to
# Example: median se above = 1, below = 0
df["median_house_value_class"] = (df[target_col] > df[target_col].median()).astype(int)

X = df.drop(columns=[target_col, "median_house_value_class"])
y = df["median_house_value_class"]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Decision Threshold Sensitivity
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. CREATE DUMMY DATA (Replace this block with your actual dataset loading)
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. TRAIN THE ML MODEL
model = RandomForestClassifier(random_state=42)
model.fit(X_clf_train, y_clf_train)

# 3. FIX: Generate the missing probability predictions (Must extract class 1 probabilities)
y_proba_clf = model.predict_proba(X_clf_test)[:, 1]

# 4. YOUR ORIGINAL CODE (Now it will run successfully)
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
results = []

for t in thresholds:
    preds = (y_proba_clf >= t).astype(int)
    results.append([t,
                    precision_score(y_clf_test, preds),
                    recall_score(y_clf_test, preds),
                    f1_score(y_clf_test, preds)])

threshold_df = pd.DataFrame(results, columns=['Threshold','Precision','Recall','F1'])
print(threshold_df)


In [ ]:
# Regularization Experiment
# Strong regularization (C=0.01)
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Target banao
target_col = "median_house_value"
df = df.copy()
df["median_house_value_class"] = (df[target_col] > df[target_col].median()).astype(int)

X = df.drop(columns=[target_col, "median_house_value_class"])
y = df["median_house_value_class"]

# 2) Column types
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# 3) Preprocessing
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ]
)

# 4) Model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, C=0.01, class_weight="balanced"))
])

# 5) Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6) Fit
model.fit(X_train, y_train)

# 7) Predict
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 8) Result
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Bootstrap Confidence Interval for AUC Difference
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. CREATE DUMMY DATA (Replace this block with your actual dataset loading)
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. TRAIN BOTH MODELS
model_baseline = LogisticRegression(random_state=42)
model_strong = RandomForestClassifier(random_state=42)

model_baseline.fit(X_clf_train, y_clf_train)
model_strong.fit(X_clf_train, y_clf_train)

# 3. FIX: Generate the missing continuous prediction probabilities for both models
y_proba_clf = model_baseline.predict_proba(X_clf_test)[:, 1]     # Probabilities for Base model
y_proba_strong = model_strong.predict_proba(X_clf_test)[:, 1]   # Probabilities for Strong model

# =========================================================================
# 4. YOUR ORIGINAL CODE (Now it will run successfully)
# =========================================================================
n_boot = 500
diffs = []

# Make sure y_clf_test is a pandas Series or numpy array
y_test_arr = np.array(y_clf_test)
proba_base_arr = np.array(y_proba_clf)
proba_strong_arr = np.array(y_proba_strong)

for i in range(n_boot):
    idx = np.random.choice(len(y_test_arr), size=len(y_test_arr), replace=True)
    y_true_sample = y_test_arr[idx]
    proba_base = proba_base_arr[idx]
    proba_strong = proba_strong_arr[idx]

    # Skip samples where only one class is present
    if len(np.unique(y_true_sample)) < 2:
        continue

    auc_base = roc_auc_score(y_true_sample, proba_base)
    auc_strong = roc_auc_score(y_true_sample, proba_strong)
    diffs.append(auc_base - auc_strong)

diffs = np.array(diffs)

mean_diff = np.mean(diffs)
ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

print("Mean AUC difference (Base - Strong):", mean_diff)
print("95% CI:", (ci_low, ci_high))


In [ ]:
#Task 1: Decision Tree Baseline

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Unconstrained Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(y_clf_train, dt.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, dt.predict(X_test_scaled))

print("Decision Tree Baseline - Train Acc:", train_acc)
print("Decision Tree Baseline - Test Acc:", test_acc)


In [ ]:
#Task 2: Controlled Decision Tree

dt_control = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_control.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(y_clf_train, dt_control.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, dt_control.predict(X_test_scaled))

print("Controlled Decision Tree - Train Acc:", train_acc)
print("Controlled Decision Tree - Test Acc:", test_acc)


In [ ]:
#Task 3: Gini vs Entropy

dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)

dt_gini.fit(X_train_scaled, y_clf_train)
dt_entropy.fit(X_train_scaled, y_clf_train)

print("Gini Test Acc:", accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled)))
print("Entropy Test Acc:", accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled)))


In [ ]:
#Task 4: Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(y_clf_train, rf.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, rf.predict(X_test_scaled))
auc = roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:,1])

print("Random Forest - Train Acc:", train_acc)
print("Random Forest - Test Acc:", test_acc)
print("Random Forest - Test ROC-AUC:", auc)

importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(5))


In [ ]:
#Task 4a: Gradient Boosting

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_clf_train)

train_acc = accuracy_score(y_clf_train, gb.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, gb.predict(X_test_scaled))
auc = roc_auc_score(y_clf_test, gb.predict_proba(X_test_scaled)[:,1])

print("Gradient Boosting - Train Acc:", train_acc)
print("Gradient Boosting - Test Acc:", test_acc)
print("Gradient Boosting - Test ROC-AUC:", auc)


In [ ]:
#Task 4b: Feature Ablation

least_features = importances.sort_values().head(5).index
mask = ~X.columns.isin(least_features)

X_train_reduced = X_train_scaled[:, mask]
X_test_reduced = X_test_scaled[:, mask]

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

auc_full = roc_auc_score(y_clf_test, rf.predict_proba(X_test_scaled)[:,1])
auc_reduced = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:,1])

print("Full RF AUC:", auc_full)
print("Reduced RF AUC:", auc_reduced)


In [ ]:
#Task 5: Cross‑Validated Comparison

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, random_state=42)

models = {
    "Logistic Regression": log_reg,
    "Decision Tree (Controlled)": dt_control,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring='roc_auc')
    results.append([name, scores.mean(), scores.std()])

cv_df = pd.DataFrame(results, columns=['Model','Mean AUC','Std AUC'])
print(cv_df)


In [ ]:
#Task 6: Hyperparameter Tuning

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_clf_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)
best_pipeline = grid.best_estimator_


In [ ]:
#Task 7: Manual Learning Curve

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []

for f in fractions:
    n = int(f * len(X_train))
    X_subset = X_train.iloc[:n]
    y_subset = y_train.iloc[:n]

    best_pipeline.fit(X_subset, y_subset)

    train_auc = roc_auc_score(y_subset, best_pipeline.predict_proba(X_subset)[:,1])
    test_auc = roc_auc_score(y_test, best_pipeline.predict_proba(X_test)[:,1])

    results.append([f, train_auc, test_auc])

lc_df = pd.DataFrame(results, columns=['Training fraction','Training AUC','Test AUC'])
print(lc_df)


In [ ]:
#Task 8: Serialize Best Model

import joblib

joblib.dump(best_pipeline, 'best_model